# Bronze Layer - Instacart Dataset

This notebook creates Delta tables in the `big_data.bronze` schema from raw CSV files stored in `/Volumes/big_data/raw/data/instacart/`.

## Tables to Create:
1. **orders** - Customer orders with timing information
2. **products** - Product catalog with aisle and department references
3. **order_products_prior** - Products in prior orders
4. **order_products_train** - Products in training set orders
5. **departments** - Product departments
6. **aisles** - Product aisles

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
# Define paths
raw_path = "/Volumes/big_data/raw/data/instacart/"
bronze_schema = "big_data.bronze"

tables = {
  "departments":          "departments.csv",
  "aisles":               "aisles.csv",
  "products":             "products.csv",
  "orders":               "orders.csv",
  "order_products_prior": "order_products__prior.csv",
  "order_products_train": "order_products__train.csv",
}

print(f"Raw data path: {raw_path}")
print(f"Target schema: {bronze_schema}")

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
print(f"Schema {bronze_schema} ready")

In [0]:



for table_name, file_name in tables.items():
    print(f"\nProcessing: {file_name}")

    # products.csv tem aspas e vírgulas no nome por isso precisa de parsing especial
    if file_name == "products.csv":
        df = spark.read.csv(
            raw_path + file_name,
            header=True,
            inferSchema=True,
            quote='"',
            escape='"',
            multiLine=True
        )
    else:
        df = spark.read.csv(
            raw_path + file_name,
            header=True,
            inferSchema=True
        )

    df = df \
        # acho que vou mudar o nome dessa coluna para updated_at(faz mais sentido)
        .withColumn("ingestion_timestamp", current_timestamp()) \
        .withColumn("source_file", lit(file_name))

    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{bronze_schema}.{table_name}")

    print(f"   {bronze_schema}.{table_name} created")
    print(f"    Rows:  {df.count():,}")
    print(f"    Columns: {len(df.columns)}")

In [0]:
print(f" Tables in {bronze_schema}:\n")
spark.sql(f"SHOW TABLES IN {bronze_schema}").show()

print(" Row count:")
for table_name in tables.keys():
    count = spark.table(f"{bronze_schema}.{table_name}").count()
    print(f"  {table_name}: {count:,} rows")